In [58]:
import numpy as np
from sklearn.datasets import load_iris
import matplotlib.pyplot as plt
import scipy

# Introduzione alla Valutazione di Modelli Classificatori
Il dataset viene suddiviso in due sottoinsiemi disgiunti:
- **Training set**: utilizzato per addestrare i parametri del modello
- **Validation set**: utilizzato per valutare le prestazioni del modello

Sui dati di validazione, calcoliamo i Log-Likelihood Ratio secondo la formula:

$$s(x_t) = \text{llr}(x_t) = \log\frac{f_{X|C}(x_t|1)}{f_{X|C}(x_t|0)}$$

dove:
- Il numeratore rappresenta la densità condizionata della classe positiva (label 1)
- Il denominatore rappresenta la densità condizionata della classe negativa (label 0)


Le predizioni vengono ottenute dai LLR assumendo **prior uniformi** sulle classi:
$$P(C=1) = P(C=0) = \frac{1}{2}$$



#### Multivariate Gaussian Model


In [59]:
dataset = np.loadtxt('../trainData.txt', delimiter=',')
D=dataset[:,:6].T
L=dataset[:,6]

labels={
    1:"True",
    0:"False"
}

D,L

(array([[ 1.85284048, -0.21530775,  0.04653456, ..., -0.06393955,
          1.06665359, -0.47186462],
        [-0.16436898,  1.63665974, -1.16487257, ..., -0.92205024,
          3.27717415, -0.37118987],
        [ 1.09839078, -0.61831145,  0.04392191, ...,  1.65445602,
         -0.53087884, -1.04041218],
        [-0.93073689,  0.99002204, -0.41369086, ..., -1.527528  ,
          1.42338924, -0.253746  ],
        [-1.0193342 ,  0.38670252, -1.08847926, ...,  1.87609599,
         -1.11381038, -0.02120134],
        [ 1.16696309,  1.15926273, -1.37988988, ...,  0.75929015,
         -0.15047129, -1.4376229 ]], shape=(6, 6000)),
 array([1., 0., 1., ..., 1., 0., 0.], shape=(6000,)))

In [60]:
def getMu(D):
    return D.sum(axis=1)/D.shape[1]

def vRow(vet):
    return vet.reshape((1, vet.size))
def vCol(vet):
     return vet.reshape((vet.size, 1))

def getC(D):
    mu=getMu(D)
    mu=vCol(mu)
    Dc=D-mu
    return (Dc @ Dc.T)/Dc.shape[1]

def compute_Sb_Sw(D, L):
    Sb = 0
    Sw = 0
    muGlobal = vCol(D.mean(1))
    for i in np.unique(L):
        DCls = D[:, L == i]
        mu = vCol(DCls.mean(1))
        Sb += (mu - muGlobal) @ (mu - muGlobal).T * DCls.shape[1]
        Sw += (DCls - mu) @ (DCls - mu).T
    return Sb / D.shape[1], Sw / D.shape[1]
    
def split_db_2to1(D, L, seed=0):
    nTrain = int(D.shape[1]*2.0/3.0)
    np.random.seed(seed)
    idx = np.random.permutation(D.shape[1])
    idxTrain = idx[0:nTrain]
    idxTest = idx[nTrain:]
    DTR = D[:, idxTrain]
    DVAL = D[:, idxTest]
    LTR = L[idxTrain]
    LVAL = L[idxTest]
    return (DTR, LTR), (DVAL, LVAL)

In [61]:
(DTR, LTR), (DVAL, LVAL) = split_db_2to1(D, L)

In [62]:
DTR.shape

(6, 4000)

In [63]:
def get_params_MVG(DTR,LTR):

    params=[]

    for i in np.unique(LTR):
        D0=DTR[:,LTR==i]
        mu=getMu(D0)
        C=getC(D0)
        params.append((mu,C))

    return params

def get_params_Naive_Bayes(DTR,LTR):
    params=[]

    for i in np.unique(LTR):
        D0=DTR[:,LTR==i]
        mu=getMu(D0)
        C=getC(D0) * np.eye(DTR.shape[0])
        params.append((mu,C))

    return params

def get_params_Tied_Covariance(DTR,LTR):
    params=[]
    for i in np.unique(LTR):
        D0 = DTR[:,LTR==i]
        mu = getMu(D0)
        params.append(mu)
    
    _,Sw=compute_Sb_Sw(DTR,LTR)
    params.append(Sw)
    return params


In [64]:
def logpdf_GAU_ND(x, mu, C):
    M=x.shape[0]
    XC = x - mu
    sign, logdet = np.linalg.slogdet(C)
    invC = np.linalg.inv(C)
    quad = np.sum(XC * (invC @ XC), axis=0)
    return -0.5 * (M * np.log(2*np.pi) + logdet + quad)

In [65]:
def getS(DVAL,params):
    res=[]

    for i in range(len(params)):
        mu=np.array(params[i][0]).reshape(DVAL.shape[0],1)
        res.append(logpdf_GAU_ND(DVAL,mu,params[i][1]))
    
    logS=np.stack(res)
    return logS

def getS_Tied_Covariance(DVAL,params):
    res=[]

    for i in range(len(params)-1):
        mu=np.array(params[i]).reshape(DVAL.shape[0],1)
        res.append(logpdf_GAU_ND(DVAL,mu,params[len(params)-1]))
    
    logS=np.stack(res)
    S=np.exp(logS)
    return S


Vediamo ora i vari classificatori ma nel caso binario dal momento in cui le etichette sono True, False, basandoci quindi sulla formula prima proposta della LLR per poi classificare in base al valore soglia t.

## Classificatore Gaussiano Multivariato

Il primo modello che implementiamo è il Classificatore Gaussiano Multivariato (MVG). Come abbiamo visto, il classificatore assume che i campioni di ogni classe c ∈ {0, 1, 2} possono essere modellati come campioni di una distribuzione gaussiana multivariata con media e matrici di covarianza dipendenti dalla classe:

$$f_{X|C}(x|c) = \mathcal{N}(x|\mu_c, \Sigma_c)$$

La soluzione ML per i parametri è data dalla media empirica e dalla matrice di covarianza di ogni classe:

$$\mu_c^* = \frac{1}{N_c} \sum_i x_{c,i}, \quad \Sigma_c^* = \frac{1}{N_c} \sum_i (x_{c,i} - \mu_c^*)(x_{c,i} - \mu_c^*)^T$$

dove $x_{c,i}$ è l'i-esimo campione della classe c.

In [66]:
def MVG_Classifier(DTR,LTR,DVAL,LVAL):
    params=get_params_MVG(DTR,LTR)
    S_MVG=getS(DVAL,params)
    log_false=S_MVG[0]
    log_true=S_MVG[1]
    s=log_true-log_false
    predictions_MVG=np.where(s>=0,1,0)
    accuracy=np.where(predictions_MVG == LVAL,1,0).mean()
    err=1-accuracy
    print(f"The error rate for Binary Classifier based on MVG is: {err*100}% and the accuracy: {accuracy*100}%")

    return predictions_MVG

predictions_MVG=MVG_Classifier(DTR,LTR,DVAL,LVAL)

The error rate for Binary Classifier based on MVG is: 6.999999999999995% and the accuracy: 93.0%


## Naive Bayes Gaussian Classifier

Consideriamo ora la versione Naive Bayes del classificatore. Come abbiamo visto, la versione Naive Bayes dell'MVG è semplicemente un classificatore Gaussiano dove le matrici di covarianza sono diagonali. La soluzione ML per i parametri della media è la stessa, mentre la soluzione ML per le matrici di covarianza è:

$$\text{diag}(\Sigma_c^*) = \text{diag}\left(\frac{1}{N_c} \sum_i (x_{c,i} - \mu_c^*)(x_{c,i} - \mu_c^*)^T\right)$$

cioè la diagonale della soluzione ML per il modello MVG. Il classificatore Naive Bayes assume che le feature siano condizionatamente indipendenti data la classe, il che semplifica il calcolo della probabilità e spesso migliora la generalizzazione quando il numero di campioni di training è limitato.

In [67]:
def Naive_Bayes_Classifier(DTR,LTR,DVAL,LVAL):
    params_Naive_Bayes=get_params_Naive_Bayes(DTR,LTR)

    S_NaiveBayes=getS(DVAL,params_Naive_Bayes)
    log_false=S_NaiveBayes[0]
    log_true=S_NaiveBayes[1]
    s=log_true-log_false
    predictions_Naive_Bayes=np.where(s>=0,1,0)
    accuracy=np.where(predictions_Naive_Bayes == LVAL,1,0).mean()
    err=1-accuracy

    print(f"The error rate for Binary Classifier based on Naive Bayes is: {err*100}% and the accuracy: {accuracy*100}%")
    return predictions_Naive_Bayes

predictions_Naive_Bayes=Naive_Bayes_Classifier(DTR,LTR,DVAL,LVAL)

The error rate for Binary Classifier based on Naive Bayes is: 7.199999999999996% and the accuracy: 92.80000000000001%


## Classificatore Gaussiano a Covarianza Legata

Consideriamo ora la versione a covarianza legata (Tied Covariance) del classificatore. In questo caso, le matrici di covarianza delle classi sono legate, con $\Sigma_c = \Sigma$. Abbiamo visto che la soluzione ML per le medie delle classi rimane la stessa. La soluzione ML per la matrice di covarianza è data dalla matrice di covarianza entro-classe empirica:

$$\Sigma^* = \frac{1}{N} \sum_c \sum_i (x_{c,i} - \mu_c^*)(x_{c,i} - \mu_c^*)^T$$

Questo approccio raggruppa le informazioni di covarianza su tutte le classi, il che può migliorare la generalizzazione quando il numero di campioni di training è limitato e riduce il numero di parametri da stimare.

In [68]:
def Tied_Covariance_Classifier(DTR,LTR,DVAL,LVAL):
    params_TCG=get_params_Tied_Covariance(DTR,LTR)
    S_TGC=getS_Tied_Covariance(DVAL,params_TCG)
    log_false=S_TGC[0]
    log_true=S_TGC[1]
    s=log_true-log_false
    predictions_TCG=np.where(s>=0,1,0)
    accuracy=np.where(predictions_TCG == LVAL,1,0).mean()
    err=1-accuracy

    print(f"The error rate for Binary Classifier based on Tied Covariance Matrix is: {err*100}% and the accuracy: {accuracy*100}%")
    return  predictions_TCG

predictions_TCG=Tied_Covariance_Classifier(DTR,LTR,DVAL,LVAL)

The error rate for Binary Classifier based on Tied Covariance Matrix is: 9.299999999999997% and the accuracy: 90.7%


In [69]:
params=get_params_MVG(DTR,LTR)
for i in range(len(params)):
    C = params[i][1]
    Corr = C / (vCol(C.diagonal()**0.5) * vRow(C.diagonal()**0.5))
    print(f"\n{'='*50}")
    print(f"Class {i} ({labels[i]})")
    print(f"{'='*50}")
    print(f"Covariance Matrix:\n{C}\n")
    print(f"Correlation Matrix:\n{Corr}\n")
   


Class 0 (False)
Covariance Matrix:
[[ 6.00956506e-01  5.15866517e-05  1.90589145e-02  1.92529876e-02
   1.28039402e-02 -1.34721598e-02]
 [ 5.15866517e-05  1.44722543e+00 -1.61340110e-02 -1.58561474e-02
  -2.64529141e-02  2.29139833e-02]
 [ 1.90589145e-02 -1.61340110e-02  5.65348901e-01 -1.84344435e-03
  -6.91446277e-03  1.68928322e-02]
 [ 1.92529876e-02 -1.58561474e-02 -1.84344435e-03  5.41615202e-01
   5.25171375e-03  1.35717775e-02]
 [ 1.28039402e-02 -2.64529141e-02 -6.91446277e-03  5.25171375e-03
   6.96067641e-01  1.58438399e-02]
 [-1.34721598e-02  2.29139833e-02  1.68928322e-02  1.35717775e-02
   1.58438399e-02  6.86519710e-01]]

Correlation Matrix:
[[ 1.00000000e+00  5.53156127e-05  3.26977873e-02  3.37466904e-02
   1.97968638e-02 -2.09743833e-02]
 [ 5.53156127e-05  1.00000000e+00 -1.78367604e-02 -1.79095288e-02
  -2.63560127e-02  2.29882544e-02]
 [ 3.26977873e-02 -1.78367604e-02  1.00000000e+00 -3.33139656e-03
  -1.10223563e-02  2.71155043e-02]
 [ 3.37466904e-02 -1.79095288e-02

Analisi feature da 0 a 3

In [70]:
DTR_sliced=DTR[0:4,:]
DVAL_sliced=DVAL[0:4,:]

predictions_MVG_sliced=MVG_Classifier(DTR_sliced,LTR,DVAL_sliced,LVAL)
predictions_Naive_Bayes_sliced=Naive_Bayes_Classifier(DTR_sliced,LTR,DVAL_sliced,LVAL)
predictions_TCG_sliced=Tied_Covariance_Classifier(DTR_sliced,LTR,DVAL_sliced,LVAL)

The error rate for Binary Classifier based on MVG is: 7.950000000000001% and the accuracy: 92.05%
The error rate for Binary Classifier based on Naive Bayes is: 7.650000000000001% and the accuracy: 92.35%
The error rate for Binary Classifier based on Tied Covariance Matrix is: 9.499999999999996% and the accuracy: 90.5%


Analisi feature 0-1 e 2-3 separatamente

In [71]:
DTR_sliced12=DTR[0:2,:]
DVAL_sliced12=DVAL[0:2,:]

predictions_MVG_sliced12=MVG_Classifier(DTR_sliced12,LTR,DVAL_sliced12,LVAL)
predictions_Naive_Bayes_sliced12=Naive_Bayes_Classifier(DTR_sliced12,LTR,DVAL_sliced12,LVAL)
predictions_TCG_sliced12=Tied_Covariance_Classifier(DTR_sliced12,LTR,DVAL_sliced12,LVAL)

The error rate for Binary Classifier based on MVG is: 36.5% and the accuracy: 63.5%
The error rate for Binary Classifier based on Naive Bayes is: 36.3% and the accuracy: 63.7%
The error rate for Binary Classifier based on Tied Covariance Matrix is: 49.45% and the accuracy: 50.55%


In [72]:
DTR_sliced34=DTR[2:4,:]
DVAL_sliced34=DVAL[2:4,:]

predictions_MVG_sliced34=MVG_Classifier(DTR_sliced34,LTR,DVAL_sliced34,LVAL)
predictions_Naive_Bayes_sliced34=Naive_Bayes_Classifier(DTR_sliced34,LTR,DVAL_sliced34,LVAL)
predictions_TCG_sliced34=Tied_Covariance_Classifier(DTR_sliced34,LTR,DVAL_sliced34,LVAL)

The error rate for Binary Classifier based on MVG is: 9.450000000000003% and the accuracy: 90.55%
The error rate for Binary Classifier based on Naive Bayes is: 9.450000000000003% and the accuracy: 90.55%
The error rate for Binary Classifier based on Tied Covariance Matrix is: 9.399999999999997% and the accuracy: 90.60000000000001%


In [73]:
def PCA(x,m):
    C=getC(x)
    s, U = np.linalg.eigh(C)
    P = U[:, ::-1][:, 0:m]
    DP = np.dot(P.T, x)
    return P,DP  # Restituisce P (matrice di proiezione)


In [74]:
# Cell: PCA preprocessing with all 3 classifiers
print("="*60)
print("CLASSIFICATION WITH PCA PREPROCESSING")
print("="*60)

for m in [1, 2, 3, 4, 5, 6]:
    print(f"\n--- m = {m} ---")
    
    # Applica PCA
    U, DTR_pca = PCA(DTR, m)
    DVAL_pca = np.dot(U.T, DVAL)
    
    # Applica i tre classificatori
    predictions_MVG_pca = MVG_Classifier(DTR_pca, LTR, DVAL_pca, LVAL)
    predictions_NB_pca = Naive_Bayes_Classifier(DTR_pca, LTR, DVAL_pca, LVAL)
    predictions_TCG_pca = Tied_Covariance_Classifier(DTR_pca, LTR, DVAL_pca, LVAL)

CLASSIFICATION WITH PCA PREPROCESSING

--- m = 1 ---
The error rate for Binary Classifier based on MVG is: 9.250000000000004% and the accuracy: 90.75%
The error rate for Binary Classifier based on Naive Bayes is: 9.250000000000004% and the accuracy: 90.75%
The error rate for Binary Classifier based on Tied Covariance Matrix is: 9.350000000000003% and the accuracy: 90.64999999999999%

--- m = 2 ---
The error rate for Binary Classifier based on MVG is: 8.799999999999997% and the accuracy: 91.2%
The error rate for Binary Classifier based on Naive Bayes is: 8.850000000000001% and the accuracy: 91.14999999999999%
The error rate for Binary Classifier based on Tied Covariance Matrix is: 9.250000000000004% and the accuracy: 90.75%

--- m = 3 ---
The error rate for Binary Classifier based on MVG is: 8.799999999999997% and the accuracy: 91.2%
The error rate for Binary Classifier based on Naive Bayes is: 8.999999999999996% and the accuracy: 91.0%
The error rate for Binary Classifier based on Tied